In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb
%run stdPlotting.ipynb
os.chdir(d['py'])
import mCGE

C:\Users\qnz630\Anaconda3\envs\GAMS46\Lib\site-packages\gams\core\numpy\gams2numpy.py:73: UserWarning: The GAMS version (46.2.0) differs from the API version (49.6.0).
  ws = GamsWorkspace()


In [2]:
d['figs'] = os.path.join(d['curr'],'results')

# Plots

Load model + data from shocks:

In [3]:
# model:
t0 = 2019
name = f'vGRSIntRC{t0}CGE'
#M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
#ws = M.ws 
# Data:
with open(os.path.join(d['curr'],'results','GRSIntRC_shocks'), "rb") as file:
    shocks = pickle.load(file)

Settings for plotting:

In [4]:
slides = True
tPlot_1 = 2030 # if one year in plot, use this
tPlot = pd.Index(range(t0, 2031), name = 't') # if multiple years, use this

Experiments names:

In [5]:
shockNames = {'Baseline': 'Baseline',
              'IRRall': 'Experiment 1',
              'IRRplastic': 'Experiment 2',
              'RCEff': 'Experiment 3',
              'taxVirgin': 'Experiment 4', 
              'taxWasteGen': 'Experiment 5',
              'RCmandate': 'Experiment 6',
              'subsidyRecycledInputs': 'Experiment 7'}

## 1. Compare CR and virgin use

Plot total CR across experiments:

In [6]:
CR_tot = pd.Series({shockNames[k]: shocks[k]['CircularRateTot'].xs(tPlot_1) for k,v in shockNames.items()})
ΔCR_tot = CR_tot[1:]-CR_tot['Baseline']

*Version 1:* Circularity rate with baseline as solid line.

In [7]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8));
CR_tot[1:].plot.bar(ax = ax);
ax.axhline(CR_tot['Baseline'], color = 'k', linewidth = 1.5);
ax.set_ylabel('Circularity rate');
fig.tight_layout();
fig.savefig(os.path.join(d['figs'],f"CRTotAll_v1.pdf"),edgecolor='k')

*Version 2:* Change in CR:

In [8]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8));
ΔCR_tot.plot.bar(ax = ax);
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
ax.set_ylabel('Δ Circularity rate');
fig.tight_layout();
fig.savefig(os.path.join(d['figs'],f"CRTotAll_v2.pdf"),edgecolor='k')

Plot relative change in virgin materials in a similar plot:

In [9]:
ΔQv_tot = pd.Series({shockNames[k]: shocks[k]['ΔQvTot_QvTot'].xs(tPlot_1) for k,v in shockNames.items() if k != 'Baseline'})

In [10]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8));
ΔQv_tot.plot.bar(ax = ax);
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5);
ax.set_ylabel('$\\Delta Q_v / Q_v$');
fig.tight_layout();
fig.savefig(os.path.join(d['figs'],f"DeltaQvTotAll_v2.pdf"),edgecolor='k')

## Experiment 1 - Improvement of internal recycling efficiency for all materials

**Short description:** The internal recycling efficiency is increased by 10 percentage points for all materials simultaneously. 

Total rebound effect over time:

In [11]:
shockId = 'IRRall'
di = shocks[shockId]

In [12]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [13]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp1_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [14]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [15]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [16]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [17]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [18]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [19]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [20]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp1_rp.pdf"),edgecolor='k')

## Experiment 2 -  Improvement of internal recycling efficiency for one material type

**Short description:** The internal recycling efficiency is increased by 10 percentage points for one material type. 

Total rebound effect over time:

In [21]:
shockId = 'IRRplastic'
di = shocks[shockId]

In [22]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [23]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp2_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [24]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [25]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [26]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [27]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [28]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [29]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [30]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp2_rp.pdf"),edgecolor='k')

## Experiment 3 -  Improvement of internal recycling efficiency for one material type

**Short description:** The recycling efficiency in the solid waste management sector is increased for all materials simultaneously. 

Total rebound effect over time:

In [31]:
shockId = 'RCEff'
di = shocks[shockId]

In [32]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [33]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp3_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [34]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [35]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [36]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [37]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [38]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [39]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [40]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp3_rp.pdf"),edgecolor='k')

## Experiment 4 -  Tax on virgin material use

**Short description:** An ad-valorem tax of 10 percentage points is introduced on all virgin materials. 

Total rebound effect over time:

In [41]:
shockId = 'taxVirgin'
di = shocks[shockId]

In [42]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [43]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp4_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [44]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [45]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [46]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [47]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [48]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [49]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [50]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp4_rp.pdf"),edgecolor='k')

## Experiment 5 - Waste tax

**Short description:** A tax on waste is introduced for all sectors and all waste types. The tax is based on mass. 

Total rebound effect over time:

In [51]:
shockId = 'taxWasteGen'
di = shocks[shockId]

In [52]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [53]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp5_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [54]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [55]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [56]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [57]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [58]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [59]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [60]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp5_rp.pdf"),edgecolor='k')

## Experiment 6 - Mandate to increase recycling rate

**Short description:** A mandate to increase recycling of textiles in the solid waste management sector.

Total rebound effect over time:

In [61]:
shockId = 'RCmandate'
di = shocks[shockId]

In [62]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [63]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp6_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [64]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [65]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [66]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [67]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [68]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [69]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [70]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp6_rp.pdf"),edgecolor='k')

## Experiment 7 - Subsidy to recycled materials

**Short description:** Introducing a subsidy to recycled material use in all production sectors. 

Total rebound effect over time:

In [71]:
shockId = 'subsidyRecycledInputs'
di = shocks[shockId]

In [72]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,6))
seaborn.lineplot(data = adj.rc_pd(di['RbvTot'], tPlot), linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_ReboundTot.pdf"),edgecolor='k')

Change in material consumption over time:

In [73]:
%%capture
largeFont()
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
df = pd.concat([
    di[f'ΔQ{k}Tot_Q{k}Tot'] for k in ('v', 'r', 't')
], axis=1) * 100
df.columns = ["Virgin", "Recycled", "Total"]
df = adj.rc_pd(df, tPlot)
seaborn.lineplot(data=df, linewidth=3, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('% change')
ax.legend(loc='upper center', frameon=True, ncol=3, bbox_to_anchor=(.5, 1.2))
ax.axhline(0, color='k', linewidth=1, alpha=.5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'], f"Exp7_MaterialTot.pdf"), edgecolor='k')

Rebound effect over $m$:

In [74]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,7))
df = adj.rc_pd(di['Rbv'], tPlot).unstack('m').rename_axis(None, axis =1)
seaborn.lineplot(data = df, linewidth = 3, ax = ax);
ax.set_xlabel('');
ax.set_ylabel('Rebound rate');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(1, color = 'k', linewidth = 1)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_Rebound.pdf"),edgecolor='k')

Change in material consumption across $m$:

In [75]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
df = (pd.concat([di[f'ΔQ{k}_Q{k}'].rename(f'$\\Delta Q_{k}/Q_{k}$') for k in ('v','r','t')], axis = 1)*100).xs(tPlot_1)
df.columns = ["Virgin", "Recycled", "Total"]
df.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 3, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_Material.pdf"),edgecolor='k')

Change in domestic sector composition in 2030: Not a lot happens here it seems...

In [76]:
df = pd.concat([shock_i['RealGDPi'].xs(tPlot_1)/shock_i['RealGDPi'].xs(tPlot_1).sum() for shock_i in (shocks['Baseline'],di)], axis = 1)
Δyi = df.diff(axis=1)[1]

Change in relative prices, virgin / recycled (if no domestic price, use foreign price):

In [77]:
def getPrice(mp, k):
    return mp.xs((k,'D'), level = ('k','o')).combine_first(mp.xs((k,'F'), level = ('k','o')))
mp_base = pd.concat([getPrice(shocks['Baseline']['MaterialPrices'], 'V').rename('V'), 
                     getPrice(shocks['Baseline']['MaterialPrices'], 'R').rename('R')], axis = 1)
mp_shock= pd.concat([getPrice(di['MaterialPrices'], 'V').rename('V'), 
                     getPrice(di['MaterialPrices'], 'R').rename('R')], axis = 1)

Percentage change in material prices and virgin/recycled ratio:

In [78]:
Δmp = (mp_shock.xs(tPlot_1)/mp_base.xs(tPlot_1)-1)*100
Δrp = ((mp_shock.xs(tPlot_1)['V']/mp_shock.xs(tPlot_1)['R']) / (mp_base.xs(tPlot_1)['V']/mp_base.xs(tPlot_1)['R']) - 1)*100

% change in material prices:

In [79]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δmp.plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_mp.pdf"),edgecolor='k')

% change in virgin/recycled ratio:

In [80]:
%%capture
largeFont()
fig, ax = plt.subplots(1,1, figsize = (10,8))
Δrp.rename('$p_{m,V}/p_{m,R}$').plot.bar(ax = ax);
ax.set_xlabel('');
ax.set_ylabel('% change');
ax.legend(loc = 'upper center', frameon = True, ncol = 2, bbox_to_anchor= (.5,1.2));
ax.axhline(0, color = 'k', linewidth = 1, alpha = .5)
fig.tight_layout()
fig.savefig(os.path.join(d['figs'],f"Exp7_rp.pdf"),edgecolor='k')